# Matrix Computations in Linear Algebra

## 1. Matrix Inversion: An $\mathcal{O}(N^3)$ Process

When faced with a system of linear equations $Ax = b$, the naive algebraic approach is to compute the inverse of $A$ and find $x = A^{-1}b$. However, explicit matrix inversion is computationally expensive and numerically unstable.

### 1.1 Why is Explicit Inversion Numerically Unstable?

The numerical instability of explicit matrix inversion stems from the accumulation of floating-point round-off errors and the amplification of these errors by the matrix's **condition number**, $\kappa(A) = \|A\| \|A^{-1}\|$.

When computing $A^{-1}$, we are essentially solving $Ax_i = e_i$ for every standard basis vector $e_i$. In floating-point arithmetic, the computed inverse $(A^{-1})_{\text{comp}}$ satisfies an error bound proportional to the condition number:

$$
\frac{\| (A^{-1})_{\text{comp}} - A^{-1} \|}{\|A^{-1}\|} \approx \mathcal{O}(\epsilon_{\text{mach}} \kappa(A))
$$

where $\epsilon_{\text{mach}}$ is the machine precision.

**Derivation of this Error Bound:**

To understand where this bound comes from, let $X = A^{-1}$ be the exact right inverse, meaning $AX = I$. Let $\tilde{X} = X + \delta X$ be our computed inverse, which is the exact inverse of some slightly perturbed matrix $A + \delta A$ due to floating-point representation and operation errors. Thus, we have:

$$
(A + \delta A)(X + \delta X) = I
$$

Expanding this product yields:

$$
AX + A(\delta X) + (\delta A)X + (\delta A)(\delta X) = I
$$

Since $AX = I$, we can subtract $I$ from both sides. Assuming the errors are small, we can drop the second-order error term $(\delta A)(\delta X) \approx 0$, leaving:

$$
A(\delta X) + (\delta A)X \approx 0 \implies A(\delta X) \approx -(\delta A)X
$$

Multiply both sides on the left by $A^{-1}$:

$$
\delta X \approx -A^{-1}(\delta A)X
$$

Taking the matrix norm of both sides and using the submultiplicative property ($\|AB\| \le \|A\|\|B\|$):

$$
\|\delta X\| \le \|A^{-1}\| \|\delta A\| \|X\|
$$

Now, divide both sides by $\|X\|$ to find the relative error of the inverse:

$$
\frac{\|\delta X\|}{\|X\|} \le \|A^{-1}\| \|\delta A\|
$$

To introduce the condition number, we multiply and divide the right side by $\|A\|$:

$$
\frac{\|\delta X\|}{\|X\|} \le (\|A^{-1}\| \|A\|) \frac{\|\delta A\|}{\|A\|} = \kappa(A) \frac{\|\delta A\|}{\|A\|}
$$

In standard floating-point arithmetic, the relative backward error introduced by the machine operations on $A$ is bounded by the machine precision, so $\frac{\|\delta A\|}{\|A\|} = \mathcal{O}(\epsilon_{\text{mach}})$. Substituting $\tilde{X} = (A^{-1})_{\text{comp}}$ and $X = A^{-1}$ yields our final result:

$$
\frac{\| (A^{-1})_{\text{comp}} - A^{-1} \|}{\|A^{-1}\|} \approx \mathcal{O}(\epsilon_{\text{mach}} \kappa(A))
$$

When we subsequently compute the matrix-vector product $x = (A^{-1})_{\text{comp}} b$, this intermediate dense matrix multiplication introduces further round-off errors. More importantly, forming $A^{-1}$ explicitly often creates extremely large intermediate values that destroy significant digits due to catastrophic cancellation.

Direct methods like LU decomposition solve $Ax = b$ without forming the intermediate matrix $A^{-1}$, thereby exhibiting much better backward error stability.

***Example:*** Consider the following ill-conditioned $2 \times 2$ matrix with a small parameter $\epsilon$ (e.g., $\epsilon = 10^{-4}$):

$$
A = \begin{pmatrix} 1 & 1 \\\\ 1 & 1 + \epsilon \end{pmatrix}
$$

Suppose we want to solve $Ax = b$ where $b = \begin{pmatrix} 2 \\\\ 2 \end{pmatrix}$. The exact solution is clearly $x = \begin{pmatrix} 2 \\\\ 0 \end{pmatrix}$.

The exact inverse of $A$ is:

$$
A^{-1} = \frac{1}{\epsilon} \begin{pmatrix} 1 + \epsilon & -1 \\\\ -1 & 1 \end{pmatrix} = \begin{pmatrix} 1 + \frac{1}{\epsilon} & -\frac{1}{\epsilon} \\\\ -\frac{1}{\epsilon} & \frac{1}{\epsilon} \end{pmatrix}
$$

Notice the terms $\frac{1}{\epsilon}$. Because $\epsilon$ is very small, these entries become extremely large.
Now, suppose during the storage of $b$ or a previous computation, a tiny floating-point round-off error $\delta$ is introduced such that our input becomes $b_{\text{noisy}} = \begin{pmatrix} 2 \\\\ 2 + \delta \end{pmatrix}$.

Using explicit inversion to find the new solution $x_{\text{noisy}}$:

$$
x_{\text{noisy}} = A^{-1} b_{\text{noisy}} = \begin{pmatrix} 1 + \frac{1}{\epsilon} & -\frac{1}{\epsilon} \\\\ -\frac{1}{\epsilon} & \frac{1}{\epsilon} \end{pmatrix} \begin{pmatrix} 2 \\\\ 2 + \delta \end{pmatrix}
$$

$$
x_{\text{noisy}} = \begin{pmatrix} 2\left(1 + \frac{1}{\epsilon}\right) - \frac{2+\delta}{\epsilon} \\\\ -\frac{2}{\epsilon} + \frac{2+\delta}{\epsilon} \end{pmatrix} = \begin{pmatrix} 2 - \frac{\delta}{\epsilon} \\\\ \frac{\delta}{\epsilon} \end{pmatrix}
$$

If our system parameter is $\epsilon = 10^{-4}$ and our microscopic measurement/rounding error is $\delta = 10^{-4}$, the calculated solution becomes $x_{\text{noisy}} = \begin{pmatrix} 1 \\\\ 1 \end{pmatrix}$.

**A tiny** $0.005\%$ **error in the input vector** $b$ **resulted in a massive, fundamentally incorrect solution!** Explicitly forming the inverse forces the computer to multiply by these massive $\frac{1}{\epsilon}$ elements, maximizing the destructive impact of small errors and demonstrating why $x = A^{-1}b$ is computationally delicate.

Let us formally analyze the computational complexity of standard matrix inversion via Gauss-Jordan elimination for an $N \times N$ matrix.

### 1.2 $\mathcal{O}(N^3)$ Complexity

In Gauss-Jordan elimination, we augment the matrix $A$ with the identity matrix $I$: $[A | I]$, and apply elementary row operations to transform $A$ into $I$, yielding $[I | A^{-1}]$.

1. **Pivot Selection:** For each of the $N$ columns, we select a pivot (often on the diagonal, $a_{kk}$).

2. **Row Normalization:** We divide the $k$-th row by the pivot $a_{kk}$. This requires $\sim 2N$ operations (since the augmented row has length $2N$).

3. **Elimination:** For every other row $i \neq k$ (there are $N-1$ such rows), we subtract a multiple of the $k$-th row to make the $i$-th entry in the $k$-th column zero.

   $$
   \text{Row}_i \leftarrow \text{Row}_i - a_{ik} \text{Row}_k
   $$

   This row operation involves a multiplication and a subtraction for each of the $2N$ elements, requiring $2 \times 2N = 4N$ operations per row.
   For $N-1$ rows, this is roughly $4N^2$ operations.

**Illustration: Gauss-Jordan Elimination of a** $2 \times 2$ **Matrix**

Let's explicitly invert $A = \begin{pmatrix} 2 & 1 \\\\ 4 & 3 \end{pmatrix}$ by forming the augmented matrix $[A | I]$:

$$
\begin{pmatrix} 2 & 1 & \big| & 1 & 0 \\\\ 4 & 3 & \big| & 0 & 1 \end{pmatrix}
$$

**Step 1 (Column** $k=1$**):**

* **Row Normalization:** Divide $\text{Row}_1$ by the pivot $a_{11} = 2$.

  $$
  \begin{pmatrix} 1 & 0.5 & \big| & 0.5 & 0 \\\\ 4 & 3 & \big| & 0 & 1 \end{pmatrix}
  $$

* **Elimination:** Subtract $4 \times \text{Row}_1$ from $\text{Row}_2$ to eliminate the $4$.

  $$
  \begin{pmatrix} 1 & 0.5 & \big| & 0.5 & 0 \\\\ 0 & 1 & \big| & -2 & 1 \end{pmatrix}
  $$

**Step 2 (Column** $k=2$**):**

* **Row Normalization:** Divide $\text{Row}_2$ by the pivot $a_{22} = 1$ (already normalized).

* **Elimination:** Subtract $0.5 \times \text{Row}_2$ from $\text{Row}_1$ to eliminate the $0.5$.

  $$
  \begin{pmatrix} 1 & 0 & \big| & 1.5 & -0.5 \\\\ 0 & 1 & \big| & -2 & 1 \end{pmatrix}
  $$

The right half is now our inverse: $A^{-1} = \begin{pmatrix} 1.5 & -0.5 \\\\ -2 & 1 \end{pmatrix}$.

Since we repeat this process for all $N$ columns, the naive total number of floating-point operations (FLOPs) appears to be $\approx 4N^3$. However, by deliberately omitting arithmetic operations on known structural zeros (such as the zeroes in the original identity matrix $I$ and the zeroes progressively formed in eliminated columns of $A$), an optimized algorithm significantly reduces this load. The exact optimized complexity for matrix inversion becomes:

$$
\text{Total FLOPs} \approx 2 N^3
$$

Thus, explicit matrix inversion is an $\mathcal{O}(N^3)$ process. For large $N$, this is prohibitively slow. Therefore, numerical linear algebra relies heavily on **Matrix Factorization** to solve $Ax = b$ without ever explicitly computing $A^{-1}.


## 2. Matrix Factorization Algorithms

In computational physics, matrix factorization is indispensable for solving large-scale systems where physical conservation laws and spatial discretizations naturally lead to dense or highly structured sparse matrices. For instance, when solving the Poisson equation for electrostatic potentials, simulating fluid dynamics via the Navier-Stokes equations, or performing quantum mechanical electronic structure calculations (like Density Functional Theory), we frequently encounter massive linear systems. Instead of repeatedly calculating costly and unstable inverses during iterative time-stepping or self-consistent field cycles, we must rely on matrix factorizations to efficiently and stably update solutions, conserving both computational time and memory.

Instead of finding $A^{-1}$, we decompose $A$ into a product of simpler matrices.

### 2.1 LU Decomposition

LU decomposition factors $A$ into a Lower triangular matrix $L$ and an Upper triangular matrix $U$ such that $A = LU$. Once factored, $Ax = b$ becomes $LUx = b$, which is solved by two $\mathcal{O}(N^2)$ substitution steps:

1. Solve $Ly = b$ (forward substitution)

2. Solve $Ux = y$ (backward substitution)

#### Derivation (Doolittle's Algorithm)

Assume $A = LU$, where $L$ has 1s on the diagonal ($l_{ii} = 1$).

$$
a_{ij} = \sum_{k=1}^{\min(i,j)} l_{ik} u_{kj}
$$

**For elements of U (where** $i \le j$**):**

$$
a_{ij} = \sum_{k=1}^{i} l_{ik} u_{kj} = \left( \sum_{k=1}^{i-1} l_{ik} u_{kj} \right) + l_{ii} u_{ij}
$$

Since $l_{ii} = 1$, we isolate $u_{ij}$:

$$
u_{ij} = a_{ij} - \sum_{k=1}^{i-1} l_{ik} u_{kj}
$$

**For elements of L (where** $i > j$**):**

$$
a_{ij} = \sum_{k=1}^{j} l_{ik} u_{kj} = \left( \sum_{k=1}^{j-1} l_{ik} u_{kj} \right) + l_{ij} u_{jj}
$$

Isolating $l_{ij}$, we get:

$$
l_{ij} = \frac{1}{u_{jj}} \left( a_{ij} - \sum_{k=1}^{j-1} l_{ik} u_{kj} \right)
$$

This yields a recursive algorithm to compute all elements of $L$ and $U$ in $\frac{2}{3}N^3$ operations—still $\mathcal{O}(N^3)$, but since full inversion takes $2N^3$ operations, this factorization is exactly **3 times faster than full inversion**, and is only needed once per matrix $A$.

### 2.2 Cholesky Decomposition

If $A$ is symmetric ($A = A^T$) and positive definite ($x^T A x > 0$ for all $x \neq 0$), we can use a more efficient factorization: $A = LL^T$, where $L$ is lower triangular.

#### Derivation

From $A = LL^T$, the element $a_{ij}$ (for $i \ge j$) is given by the inner product of the $i$-th and $j$-th rows of $L$:

$$
a_{ij} = \sum_{k=1}^j l_{ik} l_{jk}
$$

**Diagonal elements (**$i = j$**):**

$$
a_{jj} = \sum_{k=1}^j l_{jk}^2 = \sum_{k=1}^{j-1} l_{jk}^2 + l_{jj}^2
$$

$$
l_{jj} = \sqrt{a_{jj} - \sum_{k=1}^{j-1} l_{jk}^2}
$$

Because $A$ is positive definite, the term under the square root is guaranteed to be strictly positive.

**Off-diagonal elements (**$i > j$**):**

$$
a_{ij} = \sum_{k=1}^{j-1} l_{ik} l_{jk} + l_{ij} l_{jj}
$$

$$
l_{ij} = \frac{1}{l_{jj}} \left( a_{ij} - \sum_{k=1}^{j-1} l_{ik} l_{jk} \right)
$$

Cholesky decomposition requires only $\frac{1}{3}N^3$ FLOPs, making it twice as fast as LU decomposition for symmetric positive definite matrices.


## 3. Special Matrices and Fast Algorithms

When matrices possess specific structures, we can bypass the $\mathcal{O}(N^3)$ bottleneck entirely.

### 3.1 Tridiagonal Matrices

A tridiagonal matrix has non-zero elements only on the main diagonal and the first diagonals above and below it.

$$
A = \begin{pmatrix}
b_1 & c_1 & 0 & \cdots & 0 \\\\
a_2 & b_2 & c_2 & \cdots & 0 \\\\
0 & a_3 & b_3 & \cdots & 0 \\\\
\vdots & \vdots & \ddots & \ddots & c_{n-1} \\\\
0 & 0 & \cdots & a_n & b_n
\end{pmatrix}
$$

#### Derivation of the Thomas Algorithm

We wish to solve $Ax = d$. The $i$-th equation is:

$$
a_i x_{i-1} + b_i x_i + c_i x_{i+1} = d_i
$$

(where $a_1 = 0$ and $c_n = 0$).

We hypothesize a forward-elimination relationship where each $x_i$ depends only on $x_{i+1}$:

$$
x_i = c'_i x_{i+1} + d'_i
$$

Substitute this into the original equation for $x_{i-1}$:

$$
a_i (c'_{i-1} x_i + d'_{i-1}) + b_i x_i + c_i x_{i+1} = d_i
$$

Group the $x_i$ terms:

$$
(a_i c'_{i-1} + b_i) x_i + c_i x_{i+1} = d_i - a_i d'_{i-1}
$$

Solve for $x_i$:

$$
x_i = \left( \frac{-c_i}{a_i c'_{i-1} + b_i} \right) x_{i+1} + \left( \frac{d_i - a_i d'_{i-1}}{a_i c'_{i-1} + b_i} \right)
$$

By comparing this to our hypothesis $x_i = c'_i x_{i+1} + d'_i$, we extract the recursive formulas:

$$
c'_i = \frac{-c_i}{b_i + a_i c'_{i-1}}
$$

$$
d'_i = \frac{d_i - a_i d'_{i-1}}{b_i + a_i c'_{i-1}}
$$

**Algorithm:**

1. **Forward Sweep:** Compute $c'_i$ and $d'_i$ from $i=1$ to $N$.

2. **Backward Substitution:** Compute $x_n = d'_n$, then iteratively find $x_i = c'_i x_{i+1} + d'_i$ for $i = N-1$ down to 1.

**Complexity:** $\mathcal{O}(N)$. We have reduced an $\mathcal{O}(N^3)$ problem to $\mathcal{O}(N)$ by exploiting sparsity.

**==> Band-diagonal matrices (bandwidth** $m$**) can similarly be solved in** $\mathcal{O}(m^2 N)$ **time.**

### 3.2 Vandermonde Matrices

A Vandermonde matrix arises in polynomial interpolation:

$$
V = \begin{pmatrix}
1 & x_1 & x_1^2 & \cdots & x_1^{n-1} \\\\
1 & x_2 & x_2^2 & \cdots & x_2^{n-1} \\\\
\vdots & \vdots & \vdots & \ddots & \vdots \\\\
1 & x_n & x_n^2 & \cdots & x_n^{n-1}
\end{pmatrix}
$$

#### Derivation of the Determinant

We want to prove $\det(V) = \prod_{1 \le i < j \le n} (x_j - x_i)$.
Consider $\det(V)$ as a polynomial $P(x_n)$ in the variable $x_n$. If $x_n = x_i$ for any $i < n$, two rows of the matrix become identical, meaning the determinant is 0.
Therefore, $(x_n - x_1)(x_n - x_2)\dots(x_n - x_{n-1})$ must be a factor of $\det(V)$.
By expanding the determinant along the last row, the coefficient of $x_n^{n-1}$ is precisely the determinant of the $(n-1) \times (n-1)$ Vandermonde submatrix.
By induction, this implies the determinant is the product of all pairwise differences:

$$
\det(V) = \prod_{1 \le i < j \le n} (x_j - x_i)
$$

**Inversion of** $V$ **takes** $\mathcal{O}(N^2)$ **using the Parker-Traub algorithm, exploiting this polynomial structure.**

### 3.3 Toeplitz Matrices

A Toeplitz matrix has constant diagonals ($T_{i,j} = t_{i-j}$):

$$
T = \begin{pmatrix}
t_0 & t_{-1} & t_{-2} & \cdots \\\\
t_1 & t_0 & t_{-1} & \cdots \\\\
t_2 & t_1 & t_0 & \cdots \\\\
\vdots & \vdots & \vdots & \ddots
\end{pmatrix}
$$

These arise fundamentally in signal processing when dealing with wide-sense stationary (WSS) random processes. A key property of a WSS process is that its statistical properties are invariant to shifts in time. Consequently, the autocorrelation function $R(\tau) = \mathbb{E}[x(t)x(t+\tau)]$ depends only on the time lag $\tau$, not on the absolute time $t$. When we sample this process to create a discrete autocorrelation matrix $R$ for a sequence of $N$ samples, its elements are given by $R_{i,j} = R(|i-j|)$. Because the value of any matrix element depends purely on the difference between its row and column indices, the matrix is strictly Toeplitz.

Due to their shift-invariant structure, linear systems $Tx = b$ can be solved in $\mathcal{O}(N^2)$ using the **Levinson-Durbin recursion**, which iteratively builds the solution of an $n \times n$ system from the $(n-1) \times (n-1)$ system.

#### Example: Gravitational Wave Astronomy

A striking application of this structure is found in Gravitational Wave (GW) astronomy (such as LIGO/Virgo data analysis). GW detectors produce a data stream $d(t)$ containing a potential GW signal $h(t)$ buried in immense detector noise $n(t)$. To detect the signal via matched filtering, astrophysicists must compute the inner product of the data with a template, weighted by the inverse of the noise covariance matrix.

Assuming the detector noise is **stationary** over the short observation window, the time-domain noise covariance matrix $C$ has elements defined by the expectation value $C_{ij} = \mathbb{E}[n(t_i) n(t_j)]$. Because the noise is stationary, this correlation depends solely on the time lag $|t_i - t_j|$ and not on the value of time (since its characteristics are time-translation invariant), naturally **making** $C$ **a dense Toeplitz matrix**.

<!-- By the **Wiener-Khinchin theorem**, the autocorrelation function $R(\tau)$ of a wide-sense stationary process and its Power Spectral Density (PSD), $S(f)$, form a Fourier transform pair. Specifically, the autocorrelation is the continuous inverse Fourier transform of the PSD:

$$
R(\tau) = \int_{-\infty}^{\infty} S(f) e^{i 2 \pi f \tau} df
$$ -->

<!-- This theorem directly implies our mathematical link: because the elements of our discrete noise covariance matrix $C$ are constructed precisely by sampling this autocorrelation function at discrete time lags $\tau = |t_i - t_j|$, the Toeplitz covariance matrix $C$ is simply the discretely sampled inverse Fourier transform of the noise PSD. Exploiting this Toeplitz structure allows scientists to efficiently invert massive noise matrices—or compute the matched filter directly in the frequency domain—bypassing an otherwise impossible $\mathcal{O}(N^3)$ computational bottleneck. -->



## 4. Spectral Decompositions and Compression

### 4.1 Eigenvalue Problems

For a square matrix $A$, a scalar $\lambda$ and vector $v$ satisfying $Av = \lambda v$ are an eigenvalue and eigenvector. The set of all eigenvalues is the *spectrum*. If $A$ has $N$ linearly independent eigenvectors, we can diagonalize it: $A = Q \Lambda Q^{-1}$, where $\Lambda$ is diagonal.

### 4.2 Singular Value Decomposition (SVD)

SVD is a generalization of diagonalization that applies to *any* $m \times n$ matrix $A$.

$$
A = U \Sigma V^T
$$

Where $U$ is an $m \times m$ **orthogonal** matrix, $V$ is an $n \times n$ **orthogonal** matrix, and $\Sigma$ is an $m \times n$ **diagonal** matrix of singular values ($\sigma_i \ge 0$).

#### Derivation and Construction of SVD

1. Consider the matrix $A^T A$. This is an $n \times n$ symmetric, positive semi-definite matrix.

2. By the Spectral Theorem, $A^T A$ has real, non-negative eigenvalues $\lambda_i$ and a set of orthonormal eigenvectors $v_i$.

   $$
   (A^T A) v_i = \lambda_i v_i
   $$

3. Let $\sigma_i = \sqrt{\lambda_i}$. These are the **singular values**. The matrix $V$ is formed by columns $v_i$: $V = [v_1, v_2, \dots, v_n]$.

4. We define the vectors $u_i$ as:

   $$
   u_i = \frac{1}{\sigma_i} A v_i \quad \text{for } \sigma_i > 0
   $$

5. Let us check if the $u_i$ vectors are orthonormal:

   $$
   u_i^T u_j = \left( \frac{1}{\sigma_i} A v_i \right)^T \left( \frac{1}{\sigma_j} A v_j \right) = \frac{1}{\sigma_i \sigma_j} v_i^T A^T A v_j
   $$

   Since $v_j$ is an eigenvector of $A^T A$ with eigenvalue $\sigma_j^2$:

   $$
   u_i^T u_j = \frac{1}{\sigma_i \sigma_j} v_i^T (\sigma_j^2 v_j) = \frac{\sigma_j}{\sigma_i} v_i^T v_j
   $$

   Because the $v$ vectors are orthonormal, $v_i^T v_j = \delta_{ij}$. Thus $u_i^T u_j = \delta_{ij}$, proving $U$ is orthogonal.

6. Rewriting the definition $u_i = \frac{1}{\sigma_i} A v_i$, we get $A v_i = \sigma_i u_i$. In matrix form, this is precisely $AV = U\Sigma$, or $A = U\Sigma V^T$.

### 4.3 Matrix Compression via Truncated SVD

SVD provides an optimal way to compress data matrices (e.g., images or datasets). We can write the SVD as a sum of rank-1 matrices:

$$
A = \sum_{i=1}^r \sigma_i u_i v_i^T
$$

where $r$ is the rank of $A$.

To understand how the dimensions align here, recall that $A$ is an $m \times n$ matrix. In this summation, each $u_i$ is an $m \times 1$ column vector extracted from $U$, and each $v_i$ is an $n \times 1$ column vector extracted from $V$. The transpose $v_i^T$ makes it a $1 \times n$ row vector. When we compute the outer product $u_i v_i^T$, we multiply an $(m \times 1)$ matrix by a $(1 \times n)$ matrix. The inner dimensions cancel out perfectly, producing a full $m \times n$ matrix that has a rank of exactly 1. To see why its rank is 1, consider the columns of this new matrix: every column is just the vector $u_i$ multiplied by a scalar (the corresponding scalar entry from $v_i^T$). Since all columns are simply linear multiples of a single non-zero vector $u_i$, the dimension of the column space is 1. The scalar singular value $\sigma_i$ simply serves as a weight for this matrix. By summing these $r$ individual $m \times n$ \"building block\" matrices together, we perfectly reconstruct the original full-rank $m \times n$ matrix $A$.

By keeping only the top $k$ singular values ($k < r$), we get the truncated SVD:

$$
A_k = \sum_{i=1}^k \sigma_i u_i v_i^T
$$

**Eckart-Young-Mirsky Theorem:** $A_k$ is the absolute best rank-$k$ approximation of $A$ in terms of minimizing the Frobenius norm $\|A - A_k\|_F$. This is the mathematical foundation for Principal Component Analysis (PCA) and data compression.


## 5. Example: Normal Mode Analysis (NMA) in Molecular Vibrations

We now apply these linear algebra tools to structural biology. Proteins are complex molecular machines. To understand their large-scale functional motions (like the opening and closing of a receptor), biophysicists use Normal Mode Analysis.

### Physical Setup

Consider a protein with $N$ atoms. Its configuration is given by a $3N$-dimensional vector $q$ representing the coordinates of all atoms. Let $V(q)$ be the potential energy function of the protein (derived from empirical force fields representing chemical bonds, angles, and van der Waals forces).

### Derivation of the Hessian and Equations of Motion

Assume the protein is at a local energy minimum $q_0$. We perform a Taylor series expansion of the potential energy $V(q)$ around $q_0$:

$$
V(q) \approx V(q_0) + \nabla V(q_0)^T (q - q_0) + \frac{1}{2} (q - q_0)^T H (q - q_0)
$$

1. We can set $V(q_0) = 0$ as the energy baseline.

2. Because $q_0$ is an equilibrium point (a minimum), the gradient $\nabla V(q_0) = 0$.

3. $H$ is the $3N \times 3N$ **Hessian matrix** of second derivatives evaluated at $q_0$:

   $$
   H_{ij} = \left. \frac{\partial^2 V}{\partial q_i \partial q_j} \right|_{q=q_0}
   $$

Thus, the harmonic approximation of the potential energy is:

$$
V(\Delta q) = \frac{1}{2} \Delta q^T H \Delta q
$$

where $\Delta q = q - q_0$.

The kinetic energy $T$ of the system is:

$$
T = \frac{1}{2} \dot{\Delta q}^T M \dot{\Delta q}
$$

where $M$ is a $3N \times 3N$ diagonal mass matrix containing the masses of the atoms.

Using the Lagrangian $L = T - V$, the Euler-Lagrange equations of motion are:

$$
\frac{d}{dt} \left( \frac{\partial L}{\partial \dot{\Delta q}} \right) - \frac{\partial L}{\partial \Delta q} = 0
$$

$$
M \ddot{\Delta q} + H \Delta q = 0
$$

### The Generalized Eigenvalue Problem

We seek oscillatory solutions representing the "normal modes" of the protein:

$$
\Delta q(t) = a e^{i \omega t}
$$

where $a$ is an amplitude vector and $\omega$ is the vibrational frequency.
Taking the second derivative:

$$
\ddot{\Delta q}(t) = -\omega^2 a e^{i \omega t}
$$

Substitute this into the equations of motion:

$$
-\omega^2 M a e^{i \omega t} + H a e^{i \omega t} = 0
$$

Dividing out the exponential yields a **Generalized Eigenvalue Problem**:

$$
H a = \omega^2 M a
$$

Let $\lambda = \omega^2$. We must solve $Ha = \lambda Ma$.

### Applying Matrix Factorization to Solve

Because explicit matrix inversion ($M^{-1}H$) destroys the symmetry of the problem, we will use our matrix factorizations!

Since $M$ is a diagonal matrix of positive masses, it is symmetric and positive definite. We can compute its Cholesky decomposition (which for a diagonal matrix is simply taking the square root of the diagonal entries):

$$
M = L L^T = M^{1/2} M^{1/2}
$$

Substitute $M$ into the generalized eigenvalue problem:

$$
H a = \lambda M^{1/2} M^{1/2} a
$$

Multiply on the left by $M^{-1/2}$:

$$
M^{-1/2} H a = \lambda M^{1/2} a
$$

Insert the identity matrix $I = M^{-1/2} M^{1/2}$ between $H$ and $a$:

$$
\left( M^{-1/2} H M^{-1/2} \right) \left( M^{1/2} a \right) = \lambda \left( M^{1/2} a \right)
$$

Let $\tilde{H} = M^{-1/2} H M^{-1/2}$ (the mass-weighted Hessian) and $v = M^{1/2} a$.
We have transformed the physical system into a standard eigenvalue problem:

$$
\tilde{H} v = \lambda v
$$

Because $H$ is symmetric and $M^{-1/2}$ is symmetric, $\tilde{H}$ is a symmetric matrix. Thus, by the Spectral Theorem, we are guaranteed orthogonal eigenvectors $v_i$ and real eigenvalues $\lambda_i$.

By finding the eigenvalues and eigenvectors of $\tilde{H}$ (often using SVD algorithms since $3N \times 3N$ can be massive for large proteins), we can isolate the lowest frequency normal modes ($\omega_i = \sqrt{\lambda_i}$). These low-frequency modes perfectly map out the large-scale conformational changes of proteins, demonstrating how abstract linear algebra governs the mechanics of life at the molecular level.
